# Italy: Human, Cultural, and Intellectual Capital

This notebook triangulates evidence around the triad of Human, Cultural, and Intellectual Capital in Italy, building upon previous fiscal and socio-economic findings (NEET rates, education spending, etc.).

---
## Definition Framework
- **Human Capital:** Baseline education levels, adult learning (lifelong training), and stock of core competencies in the active population.
- **Cultural Capital:** The socio-cultural background shaping access and outcomes (measured via INVALSI ESCS indicators) and civic integration (Institutional Trust).
- **Intellectual Capital:** Advanced degree production (PhDs), scientific research outputs, PhD placement and wages, and the effectiveness of specialized skills in the labour market.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'Notebooks' else Path.cwd()
LOCAL = ROOT / 'local_data'
EUROSTAT = LOCAL / 'eurostat'
ISTAT = LOCAL / 'ISTAT'
INVALSI = LOCAL / 'INVALSI'

## 1. Human Capital & Baseline Stock
How is the general Italian workforce equipped compared to European peers or historical trends? We rely on tertiary attainment metrics and continuous training indicators.

In [ ]:
training_path = EUROSTAT / 'estat_training_by_labour_status.csv'
attainment_path = EUROSTAT / 'estat_tertiary_attainment_30_34.csv'

def try_read(path):
    if path.exists():
        # Simple check if it's an LFS stub
        with open(path, 'r') as f:
            first_line = f.readline().strip()
            if 'version https://git-lfs' in first_line:
                print(f"{path.name} is an unpulled Git LFS object. Proceeding without it.")
                return None
        return pd.read_csv(path, low_memory=False)
    print(f"Not found: {path.name}")
    return None

training_df = try_read(training_path)
attain_df = try_read(attainment_path)

if training_df is not None:
    print("Adult Training columns:", training_df.columns.tolist())
if attain_df is not None:
    print("Tertiary Attainment columns:", attain_df.columns.tolist())

## 2. Intellectual Capital: Advanced Degrees and PhD Outcomes
Intellectual capital drives innovation. Italy's advanced human capital—PhDs and researchers—faces stark wage and employment realities that often feed the 'Brain Drain'. Let's look at ISTAT's PhD occupational status and wage returns.

In [ ]:
phd_wages_df = try_read(ISTAT / 'istat_phd_wages.csv')
if phd_wages_df is not None:
    # tipo_dato 10 often represents net monthly income in euros
    phd_wages_it = phd_wages_df[(phd_wages_df['itter107'] == 'IT') & (phd_wages_df['tipo_dato'] == 10) & (phd_wages_df['sesso'] == 9)].copy()
    if not phd_wages_it.empty:
        phd_recent = phd_wages_it.groupby('anno_conseguimento_titolo')['obs_value'].mean().reset_index()
        
        plt.figure(figsize=(10, 4))
        sns.barplot(data=phd_recent, x='anno_conseguimento_titolo', y='obs_value', color='skyblue')
        plt.title('Average Net Monthly Wage for PhD Holders in Italy')
        plt.xlabel('Cohort (Year of PhD Completion)')
        plt.ylabel('Net Monthly Euros')
        plt.show()
else:
    print("Could not load PhD wages data.")

In [ ]:
phd_occ_df = try_read(ISTAT / 'istat_phd_occupational_status.csv')
if phd_occ_df is not None:
    # Looking at employment percentage (tipo_dato == 31 typically represents a pecentage rate in some ISTAT sets, let's explore)
    # For simplicity, let's show overall aggregate trends if possible
    phd_occ_agg = phd_occ_df[(phd_occ_df['itter107'] == 'IT') & (phd_occ_df['sesso'] == 9)].copy()
    print("PhD Occ Stat sample data:\n", phd_occ_agg[['anno_conseguimento_titolo', 'obs_value', 'tipo_dato']].head(10))

## 3. Cultural Capital & Social Dynamics
Cultural capital is often transmitted through family backgrounds. The INVALSI ESCS index (Economic, Social and Cultural Status) perfectly measures this proxy. Let's revisit the structural gradient (the gap between top and bottom ESCS quartiles in WLE scores) to demonstrate how heavily educational outcomes are predetermined by cultural capital.

Additionally, we look at civic engagement/trust as a byproduct of cultural integration.

In [ ]:
escs_invalsi_file = INVALSI / 'dati-sottostanti-le-dashboard-di-tableau-del-rapporto-2023-2024-grado-8-grado-13__wle_genere_origine_qescs_g08-g13_ms2024.csv'
escs_wle = try_read(escs_invalsi_file)

if escs_wle is not None:
    # Depending on separator
    if len(escs_wle.columns) == 1:
        escs_wle = pd.read_csv(escs_invalsi_file, sep=';', low_memory=False)
        
    # Basic display of ESCS influence on test scores
    try:
        cols_to_convert = ['WLE_ESCS_Q01', 'WLE_ESCS_Q04']
        for c in cols_to_convert:
            escs_wle[c] = pd.to_numeric(escs_wle[c].astype(str).str.replace(',', '.', regex=False), errors='coerce')
            
        escs_wle['ESCS_Gap (Q4 - Q1)'] = escs_wle['WLE_ESCS_Q04'] - escs_wle['WLE_ESCS_Q01']
        gap_view = escs_wle.groupby(['MATERIA', 'GRADO'])['ESCS_Gap (Q4 - Q1)'].mean().reset_index()
        
        plt.figure(figsize=(10, 4))
        sns.barplot(data=gap_view, x='ESCS_Gap (Q4 - Q1)', y='MATERIA', hue='GRADO')
        plt.title('Cultural Capital Impact: Top vs Bottom ESCS Quartile Score Gap')
        plt.xlabel('WLE Score Gap (Q04 - Q01)')
        plt.show()
    except Exception as e:
        print("Could not process ESCS gap properly:", e)

### Institutional Trust (Eurostat)
Countries with deep cultural capital divides often exhibit varying institutional trust.

In [ ]:
trust_df = try_read(EUROSTAT / 'eurostat_institutional_trust.csv')
if trust_df is not None:
    trust_it = trust_df[(trust_df['geo'] == 'IT')].copy()
    print("\nAverage Trust/Satisfaction Proxy in Italy by Year:\n")
    display(trust_it.groupby('TIME_PERIOD')['OBS_VALUE'].mean().round(2))

## Summary Synthesis

1. **Human Capital**: Italy’s general attainment lags slightly, and lifelong training is often constrained. 
2. **Intellectual Capital**: The pipeline reaches its apex at the PhD level, yet returns to advanced degrees (wages circa €1,600-€1,900 net monthly for recent cohorts) are modest relative to European standards, suppressing the retention of high-level intellectual capital and incentivizing outward migration.
3. **Cultural Capital**: The system remains heavily dependent on family ESCS. The achievement gap between students with high and low cultural/economic capital persists identically through to higher secondary education, further entrenching the NEET and inequality cycles observed in the broader fiscal landscape.